# 04 · Recover work without pretending every action is atomic

## Goal
Persist one completed deterministic calculation, reopen the session and replay the saved result.

**Status:** worked example prepared by Codex; learner understanding unassessed. No API calls.

## Setup
Run top to bottom. All examples are synthetic unless explicitly labelled as public evidence.

In [1]:
from pathlib import Path
import sys, json, tempfile
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'signal_lab').is_dir(), 'Run from the companion root or notebooks directory'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (8, 4.2), 'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})


## Steps

### 1. Register a bounded tool and persist its answer

In [2]:
from signal_lab.core import Harness, BudgetExceeded
from signal_lab.quant import executable_return
with tempfile.TemporaryDirectory() as tmp:
    path=Path(tmp)/'run.sqlite'
    h=Harness(path,'lesson',max_calls=1)
    h.register('return',executable_return)
    first=h.step('s1','return',{'entry':104,'exit':105,'cost_bps':10})
    h.close()
    h=Harness(path,'lesson',max_calls=1)
    h.register('return',lambda **kwargs: 999)
    replay=h.step('s1','return',{'entry':104,'exit':105,'cost_bps':10})
    events=h.db.execute('SELECT step,status FROM events').fetchall()
    h.close()
assert first == replay and replay != 999
pd.DataFrame(events,columns=['step','status'])

,step,status
0,s1,started
1,s1,completed


### 2. Reason about interruption
A crash after a real external side effect but before persistence can leave an ambiguous outcome. This local runner supports idempotent teaching tools; it does not promise exactly-once external writes. Prime Agent child handles likewise need explicit completion and failure handling.

In [3]:
pd.DataFrame([{'first_result':first,'replayed_result':replay,'actual_tool_attempts':sum(s=='started' for _,s in events)}])

,first_result,replayed_result,actual_tool_attempts
0,0.008615,0.008615,1


## Checks
Only one tool attempt occurred despite reopening. See tests for failures consuming budget, explicit resume and changed inputs being rejected.

## Next Steps
Optional: add one evidence-validation function to the allowed tools. Keep arbitrary shell/code execution outside this teaching runner.

**Low energy:** stop here. The checkpoint is optional; reading the worked output does not update mastery.